# Tagger A25: การประเมินด้วยข้อมูลจริง

Notebook นี้ **ไม่สร้างข้อมูลหรือคะแนนจำลอง** ทุกผลลัพธ์มาจาก `pitty/scripts/sentence_eval.py` และ `pitty/scripts/fluent_eval.py` ใน `DSI490-Sign2Say` โดยใช้ tagger ที่ deploy อยู่ใน `models/onnx/tagger.onnx` และ bank ที่กัน signer ของชุดทดสอบออกแล้ว (`artifacts/eval/bank_eval.npz`).

โปรโตคอล: ใช้ TSL51 test split ที่ล็อกไว้ก่อน (`SPLIT=test`) เพื่อวัด WER, Recall@1 และ Recall@5; จากนั้นใช้ fluent test ที่ประกอบจากคลิปคำเดี่ยวของผู้ทำท่าที่กันออกเพื่อวัดการต่อท่าลื่น. `data_test` จะรันต่อเมื่อมีวิดีโอและ ground truth อยู่จริงเท่านั้น.

In [ ]:
from pathlib import Path
import json, os, subprocess
import pandas as pd

REPO = Path('/Users/thawaphorn/Pee4/Term1/DSI490/DSI490-Sign2Say')
OUT = Path('/Users/thawaphorn/Pee4/Term1/DSI490/การทดลอง/tagger-results')
PYTHON = Path('/Users/thawaphorn/Pee4/Term1/DSI490/การทดลอง/venv-sign2say/bin/python')
OUT.mkdir(parents=True, exist_ok=True)
assert PYTHON.exists(), 'สร้าง environment ก่อน: python3 -m venv การทดลอง/venv-sign2say'
assert (REPO / 'cloud_s3/data_prep/pose_mp/tsl51_user_sentence').exists()
assert (REPO / 'artifacts/eval/bank_eval.npz').exists()
assert (REPO / 'models/onnx/tagger.onnx').exists()
print('พร้อมประเมินข้อมูลจริง')

## 1. TSL51 held-out test

คำสั่งนี้ไม่ใช้ชุด tune เพื่อเลือกค่าเพิ่ม และไม่แก้ไฟล์โมเดล production. `ORT_THREADS=8` ใช้เพียงเพิ่มความเร็ว ONNX Runtime บน CPU.

In [ ]:
result_tsl51 = OUT / 'a25_tsl51_test_1x.json'
env = dict(os.environ, MPLCONFIGDIR='/private/tmp/mpl', ORT_THREADS='8', SPLIT='test')
if not result_tsl51.exists():
    subprocess.run([str(PYTHON), 'pitty/scripts/sentence_eval.py', str(result_tsl51), '0', '1.0'], cwd=REPO, env=env, check=True)
else:
    print('ใช้ผลที่รันแล้ว:', result_tsl51)

In [ ]:
r = json.loads(result_tsl51.read_text(encoding='utf-8'))['results']
df = pd.DataFrame(r)
display(df.groupby('split')[['WER', 'recall1', 'recall5', 'n_pred', 'n_ref']].mean().round(3))
display(df[['clip', 'ref', 'top1', 'WER', 'recall1', 'recall5']].sort_values('WER', ascending=False).head(10))

## 2. Fluent test

ชุดนี้สร้างลำดับจาก landmark ของคลิปคำเดี่ยวจริงจากผู้ทำท่าที่ถูกกันออก ไม่ใช่วิดีโอประโยคธรรมชาติ จึงรายงานแยกจาก TSL51. สร้าง cache ครั้งแรกเท่านั้น แล้ววัด tagger เดิมบน split test.

In [ ]:
fluent_result = OUT / 'a25_fluent_test.json'
env = dict(os.environ, MPLCONFIGDIR='/private/tmp/mpl', ORT_THREADS='8')
if not (REPO / 'cache/fluent_eval').exists():
    subprocess.run([str(PYTHON), 'pitty/scripts/fluent_eval.py', 'build'], cwd=REPO, env=env, check=True)
if not fluent_result.exists():
    subprocess.run([str(PYTHON), 'pitty/scripts/fluent_eval.py', 'run', str(fluent_result), '--split', 'test'], cwd=REPO, env=env, check=True)
else:
    print('ใช้ผลที่รันแล้ว:', fluent_result)

In [ ]:
fr = json.loads(fluent_result.read_text(encoding='utf-8'))
display(pd.DataFrame(fr['results']).groupby('split')[['WER', 'recall1', 'recall5']].mean().round(3))

## 3. ข้อจำกัดของรอบนี้

ยังไม่มี `data_test` 5 วิดีโอพร้อมรายการคำจริง จึงไม่มีสิทธิ์คำนวณ WER หรือสรุปว่าคลิปเหล่านั้นถูก/ผิด. เมื่อมีไฟล์และ reference แล้ว ให้เพิ่ม manifest และใช้ `01_real_video_evaluation.ipynb` เพื่อสร้างภาพ top-5, probability, confidence และเวลาของแต่ละช่วงจากผลจริง.